# Model Performance

Compare saved predictions and metrics for explicit runs. Select runs, architectures,
targets, outer contexts, and the prediction role. Metrics retain target units and
selections remain separate. This notebook reads tables only; it never loads or fits
estimators. See [README.md](README.md) for research stages and interpretation.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "pipeline" / "config").exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from pipeline.modeling.api.inspection import select_artifacts
from pipeline.utils.paths import load_paths  

RUN_ROOT = load_paths().external("model_runs")


## Select saved models

Set one or more exact run IDs. Choose a role saved by every selected model:
`inner_oof`, `outer_holdout`, or `final_test`. Filters accept one or several values;
`None` selects every available model, target, or context in those runs.


In [ ]:
RUN_IDS = ["REPLACE_WITH_RUN_ID"]
RUN_DIRECTORIES = [RUN_ROOT / run_id for run_id in RUN_IDS]
ROLE = "inner_oof"
MODELS = None                 # ["ridge", "random_forest"]
TARGETS = None                # ["epg_cooling", "demand_heating"]
SELECTIONS = ["holdout_A"]   # ["holdout_A", ..., "holdout_E"] or ["final_test"]
MODEL_IDS = None              # Optional exact model ID(s)

if any("REPLACE" in str(path) for path in RUN_DIRECTORIES):
    raise ValueError(f"Set RUN_IDS to explicit run folders under {RUN_ROOT}.")
selected = select_artifacts(RUN_DIRECTORIES, role=ROLE, models=MODELS,
    targets=TARGETS, selections=SELECTIONS, model_ids=MODEL_IDS)
models, metrics = selected["models"], selected["metrics"]
predictions, folds = selected["predictions"], selected["folds"]
fold_metrics = selected["fold_metrics"]
regional_metrics = selected["regional_metrics"]
print("Prediction rows:", len(predictions))
display(models[["run_id", "model_id", "model", "target", "selection", "stage", "scope"]])


## Metric comparison

Compare architectures within each target and selection. Pooled metrics summarize
all saved predictions in the chosen role. Fold metrics are separate summaries;
their mean need not equal a pooled R-squared or error.


In [ ]:
comparison = metrics.pivot(index=["run_id", "model_id", "target", "selection", "model"],
                            columns="metric", values="value").sort_index()
display(comparison)
for target, frame in metrics.groupby("target", sort=False):
    for metric in [name for name in ["rmse", "mae", "r2", "spearman"] if name in set(frame.metric)]:
        part = frame.loc[frame.metric.eq(metric)].copy()
        part["label"] = part.run_id.str[-8:] + " / " + part.selection + " / " + part.model
        ax = part.set_index("label").value.sort_values().plot.barh(figsize=(9, max(3, .35 * len(part))))
        ax.set(title=f"{target}: {metric} ({ROLE})", xlabel=metric, ylabel="Selection / model")
        ax.grid(axis="x", alpha=.25)
        plt.tight_layout()
        plt.show()
display(metrics.loc[metrics.undefined_reason.notna()])


## Fold-specific scores and parish predictions

Residuals are observed minus predicted. Use the run and model IDs to identify the
exact fitted procedure. Inner prediction sets from different selections overlap;
never combine them as one independent OOF table.


In [ ]:
display(fold_metrics.sort_values(["target", "selection", "model", "metric", "fold"]))
display(regional_metrics.sort_values(["target", "selection", "model", "NUTS3"]))
display(predictions[["run_id", "model_id", "model", "target", "selection", "prediction_role",
                     "ID_norm", "NUTS3", "observed", "predicted", "residual"]].head(30))


## Export filtered tables

Set a destination folder to write selected model IDs, metrics, fold scores, and
predictions. Keeping IDs and selections in each table makes exported rows traceable.
Leave the folder unset to inspect without writing files.


In [ ]:
OUTPUT_DIRECTORY = None  # e.g. Path("<chosen folder>")
if OUTPUT_DIRECTORY is not None:
    output = Path(OUTPUT_DIRECTORY)
    output.mkdir(parents=True, exist_ok=True)
    for name, table in [("models", models), ("metrics", metrics), ("fold_metrics", fold_metrics),
                        ("predictions", predictions), ("regional_metrics", regional_metrics)]:
        table.to_csv(output / f"{name}.csv", index=False)
    print("Saved selected result tables to", output.resolve())
